In [1]:
import numpy as np
from ase import Atoms
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
from pathlib import Path
from scipy.interpolate import interp1d
import umap
import warnings
from abc import ABC, abstractclassmethod

from typing import List, Tuple, Union, Dict
from scipy.stats import ks_2samp
from sklearn.decomposition import PCA

## Маскирование данных для представления НЧ в базисе уникальных подструктур

In [2]:
def get_reconstruction_error_mask(weights_matrix:np.ndarray, 
                                  top_percent:float=3.0, 
                                  n_components:int=5, 
                                  seed:int=42)->np.ndarray:
    '''
    Функция возвращает top_percent индексов НЧ (weights_matrix.shape[0]) 
    в соответствии с отсортированной по убыванию ошибкой реконструкции на основе PCA.
    weights_matrix - матрица представления НЧ в базисе подструктур;
    top_percent - граница в % выбора данных для маскирования (values>(100-top_percent));
    n_components - чило компонент для PCA при расчете ошибки реконструкции;
    '''
    row_sums = weights_matrix.sum(axis=1, keepdims=True)
    row_sums[row_sums == 0] = 1.0
    normalized_weights = weights_matrix / row_sums
    
    pca = PCA(n_components, random_state=seed)
    transformed = pca.fit_transform(normalized_weights)
    reconstructed = pca.inverse_transform(transformed)

    reconstruction_errors = np.linalg.norm(normalized_weights - reconstructed, axis=1)
    
    num_items = np.ceil(reconstruction_errors.shape[0]/100*top_percent).astype(int)
    argsort = np.argsort(reconstruction_errors)

    mask = argsort[::-1][:num_items]
    return mask

def get_small_connected_components_mask(weights_matrix:np.ndarray,
                                        sub_sizes:np.ndarray=None,
                                        verbose:int=0
                                        )->np.ndarray:
    '''
    Функция возвращает индексы НЧ (weights_matrix.shape[0]) 
    представленных отдельными малыми компонентами связности в представлении weights_matrix.
    weights_matrix - не нормированная матрица представления НЧ в базисе подструктур;
    sub_sizes - массив размеров подструктур, форма должна соответствовать weights_matrix.shape[1], по умолчанию None;
    verbose - уровень подробности вывода [0,1], по умолчанию 0;
    '''
    if verbose==1:
        if sub_sizes is not None:
            assert weights_matrix.shape[1]==sub_sizes.shape[0], 'sub_sizes.shape[0] должен совпадать с weights_matrix.shape[1]'    
        np_sizes = weights_matrix.sum(axis=1).astype(int)
        isolated_substructures_report = []
        
    binary = (weights_matrix > 0).astype(float)
    adj = binary.T @ binary
    np.fill_diagonal(adj, 0)
    G = nx.from_numpy_array(adj)
    components = list(nx.connected_components(G))
    giant_component_size = G.number_of_nodes() * 0.25
    mask = np.array([]).astype(int)
    
    for idx, comp in enumerate(components):
        comp_nodes = list(comp)
        if len(comp_nodes) > giant_component_size:
            continue
        matching_particles = np.where(binary[:, comp_nodes].sum(axis=1) > 0)[0]
        mask = np.append(mask, matching_particles)

        if verbose==1:
            data = {'component_id': idx,
                    'substructure_indices': comp_nodes,
                    'num_substructures_in_comp': len(comp_nodes),
                    'associated_NPs_idx': matching_particles,
                    'associated_NPs_sizes': np_sizes[matching_particles]}
            if sub_sizes is not None:
                data['substructure_sizes'] = sub_sizes[comp_nodes]
            isolated_substructures_report.append(data)

    if verbose==1:
        for rep in isolated_substructures_report:
            print(f"\nКомпонента №{rep['component_id']} (Размер группы подструктур: {rep['num_substructures_in_comp']})")
            print(f"  -> Идексы подструктур: {rep['substructure_indices']}")
            if sub_sizes is not None:
                print(f"  -> Кол-ва атомов в подструктурах: {rep['substructure_sizes']}")
            print(f"  -> Встречается в наночастицах под индексами {rep['associated_NPs_idx']} ")
            print(f"  -> Кол-ва атомов в наночастицах {rep['associated_NPs_sizes']}")

    return mask

def get_percentile_mask(weights_matrix:np.ndarray, edge_percentile:float=5, mode:str='low'):
    '''
    Функция возвращает edge_percentile индексов НЧ (weights_matrix.shape[0]) в зависимости от размера НЧ в соответствии с mode.
    weights_matrix - матрица представления НЧ в базисе подструктур (не нормированная, сумма по axis=1 дает число атомов в НЧ);
    edge_percentile - граница в % (0-100) (при mode="high" используется 100-edge_percentile);
    mode - ограничение на нижнюю или верхнюю границу "low","high", по умолчанию "low";
    '''
    assert mode in ['low','high'], 'mode может принимать значения "low" и "high"'
    np_sizes = weights_matrix.sum(axis=1).astype(int)
    if mode=='low':
        size_threshold = np.percentile(np_sizes, edge_percentile)
        return np.where(np_sizes <= size_threshold)[0]
    else:
        size_threshold = np.percentile(np_sizes, 100-edge_percentile)
        return np.where(np_sizes >= size_threshold)[0]

In [3]:
def get_merged_mask(weights_matrix: np.ndarray, 
                    masks_config: dict, 
                    max_base_pct: float = 5.0, 
                    max_tail_pct: float = 2.0, 
                    tail_threshold: float = 0.5, 
                    verbose: int = 1
                    ) -> np.ndarray:
    '''
    Объединяет независимые маски НЧ с учетом приоритетов методов, 
    количественных ограничений и фильтрации "хвостов" для предотвращения утечки данных.
    
    weights_matrix - двумерный массив (матрица весов подструктур);
    masks_config - словарь вида {name: {"indices": [...], "priority": int}}
                   Индексы внутри "indices" упорядочены по убыванию важности если это возможно,
                   в противном случае они считаются равнозначными.
                   Выше значение "priority" -> выше приоритет метода.
    max_base_pct - максимальный процент выборки для базовой маски (0-100);
    max_tail_pct - максимальный процент выборки для хвостов (0-100);
    tail_threshold - порог вовлеченности подструктур для отбора хвостов (0.0 - 1.0) (доля подструктур в составе НЧ);
    verbose - вывод детальной информации (1 - выводить, 0 - тихий режим);
    '''
    total_objects = weights_matrix.shape[0]
    
    max_base_count = int(np.floor((max_base_pct / 100.0) * total_objects))
    max_tail_count = int(np.floor((max_tail_pct / 100.0) * total_objects))
    
    # Сортируем методы по приоритету (от высшего к низшему)
    sorted_methods = sorted(masks_config.items(), key=lambda item: item[1].get("priority", 0), reverse=True)
    
    # Каскадный отбор в базовую маску
    base_indices = []
    seen_indices = set()
    method_contributions = {}
    
    for name, config in sorted_methods:
        if len(base_indices) >= max_base_count:
            method_contributions[name] = 0
            continue
            
        added_by_method = 0
        for idx in config.get("indices", []):
            if len(base_indices) >= max_base_count:
                break
            if idx not in seen_indices:
                seen_indices.add(idx)
                base_indices.append(idx)
                added_by_method += 1
                
        method_contributions[name] = added_by_method

    # Формируем базовую маску
    base_indices_arr = np.array(base_indices, dtype=int)
    
    # Поиск индексов "хвостов" по уникальным подструктурам
    binary_matrix = (weights_matrix > 0).astype(float)
    actual_tail_indices = np.array([], dtype=int)
    total_potential_tails = 0
    ignored_tails_count = 0
    
    if len(base_indices_arr) > 0:
        # Находим подструктуры, которые уже "засветились" в базовом наборе
        edge_sub_indices = np.unique(np.where(binary_matrix[base_indices_arr] > 0)[1])
        
        if len(edge_sub_indices) > 0:
            # Считаем долю общих подструктур для ВСЕХ объектов матрицы
            sub_shares = binary_matrix[:, edge_sub_indices].sum(axis=1) / (binary_matrix.sum(axis=1) + 1e-9)
            sub_shares[base_indices_arr] = 0.0

            potential_tail_indices = np.where(sub_shares > tail_threshold)[0]
            total_potential_tails = len(potential_tail_indices)
            
            if total_potential_tails > 0:
                # Сортируем кандидатов по убыванию "важности" (по величине sub_shares)
                sort_idx = np.argsort(sub_shares[potential_tail_indices])[::-1]
                sorted_tail_candidates = potential_tail_indices[sort_idx]
                
                # Отсекаем хвосты строго по лимиту max_tail_count
                actual_tail_indices = sorted_tail_candidates[:max_tail_count]
                ignored_tails_count = max(0, total_potential_tails - max_tail_count)


    # Финальное объединение индексов
    final_indices = np.concatenate([base_indices_arr, actual_tail_indices])

    if ignored_tails_count > 0:
        massage = f"ВНИМАНИЕ: {ignored_tails_count} объектов с похожим составом подструктур \nНЕ ПОПАЛИ в Трейн из-за жесткого лимита max_tail_pct ({max_tail_pct}%)."
        warnings.warn(massage,UserWarning)
    
    if verbose == 1:
        actual_base_pct = (len(base_indices_arr) / total_objects) * 100
        actual_tail_pct = (len(actual_tail_indices) / total_objects) * 100
        total_pct = (len(final_indices) / total_objects) * 100
        
        print('=' * 80)
        print('СТАТИСТИКА НАБОРА ИНДЕКСОВ (TRAIN):')
        print(f"Всего объектов в weights_matrix: {total_objects}")
        print(f"Базовый лимит: {max_base_pct}% ({max_base_count} об.) -> Фактически набрано: {actual_base_pct:.2f}% ({len(base_indices_arr)} об.)")
        print('Вклад методов в базовый набор:')
        for name, count in method_contributions.items():
            print(f"  - {name}: {count} нов. об.")
            
        print(f"Лимит на хвосты: {max_tail_pct}% ({max_tail_count} об.)")
        print(f"Обнаружено потенциальных хвостов: {total_potential_tails} об.")
        print(f"Добавлено хвостов: {len(actual_tail_indices)} об. ({actual_tail_pct:.2f}%)")
        print(f"ИТОГОВЫЙ ОБЪЕМ ЗАРЕЗЕРВИРОВАННОГО ТРЕЙНА: {len(final_indices)} об. ({total_pct:.2f}%)")
        print('=' * 80)
        

    return final_indices

In [4]:
def get_preserved_mask(weights_matrix:np.ndarray,
                       sub_sizes:np.ndarray=None, 
                       reconstruction_n_comp:int=2, 
                       reconstruction_top_repcent:float=3.0, 
                       low_size_percentile:float=3.0, 
                       high_size_percentile:float=0.1, 
                       masks_priorities:List[int] = [3,2,1,0], 
                       max_base_percentile = 5, 
                       max_tail_percentile = 3,
                       tail_threshold = 0.65,
                       verbose:int=1, 
                       seed:int=42)->np.ndarray:
    '''
    Генерирует набор индексов НЧ для гарантированного включения в train-выборку.
    
    Функция агрегирует результаты четырех методов маскирования, каскадно объединяет их 
    в соответствии с заданными приоритетами и добирает "хвосты" (объекты со схожим составом 
    подструктур).

    Args:
        weights_matrix (np.ndarray): Двумерная матрица весов подструктур (объекты x подструктуры) (не нормированная).
        sub_sizes (np.ndarray, optional): Размеры подструктур для метода связных компонент. 
            По умолчанию None.
        reconstruction_n_comp (int): Количество компонент для метода ошибки реконструкции. 
            По умолчанию 2.
        reconstruction_top_repcent (float): Процент объектов с наибольшей ошибкой реконструкции, 
            претендующих на попадание в маску. По умолчанию 3.0.
        low_size_percentile (float): Процент объектов с наименьшим размером (кол-вом атомов). 
            Если <= 0, метод отключается. По умолчанию 3.0.
        high_size_percentile (float): Процент объектов с наибольшим размером (кол-вом атомов). 
            Если <= 0, метод отключается. По умолчанию 0.1.
        masks_priorities (List[int]): Список приоритетов для методов в следующем порядке:
            [small_connected_components, reconstruction_error, percentile_low, percentile_high].
            Чем выше число, тем выше приоритет метода при каскадном отборе. 
            По умолчанию [3,2,1,0]
        max_base_percentile (float/int): Жесткий лимит (в % от всей выборки) на объем 
            базового набора индексов, собираемого из масок. По умолчанию 5.
        max_tail_percentile (float/int): Максимальный допустимый объем (в % от всей выборки) 
            для добавления зависимых "хвостов". По умолчанию 3.
        tail_threshold (float): Порог вовлеченности подструктур (от 0.0 до 1.0). Если объект 
            состоит из подструктур, уже ушедших в базовый трейн, более чем на этот порог, 
            он помечается как "хвост". По умолчанию 0.65.
        verbose (int): Флаг для логирования. 1 — выводить статистику, 0 — тихий режим. По умолчанию 1.
        seed (int): random_seed для PCA. По умолчанию 42.

    Returns:
        np.ndarray: вектор уникальных индексов НЧ, сформированный для train-выборки.
    '''

    cc_mask = get_small_connected_components_mask(weights_matrix = weights_matrix, 
                                                  sub_sizes = sub_sizes,
                                                  verbose = verbose)
    rec_mask = get_reconstruction_error_mask(weights_matrix = weights_matrix, 
                                             top_percent = reconstruction_top_repcent, 
                                             n_components = reconstruction_n_comp, 
                                             seed = seed)
    if low_size_percentile>0:
        prc_low_mask = get_percentile_mask(weights_matrix = weights_matrix, 
                                           edge_percentile = low_size_percentile, 
                                           mode = 'low')
    else:
        prc_low_mask = np.array([])

    if high_size_percentile>0.:
        
        prc_high_mask = get_percentile_mask(weights_matrix = weights_matrix, 
                                            edge_percentile = high_size_percentile, 
                                            mode = 'high')
    else:
        prc_high_mask = np.array([])
    
    masks_config = {'small_connected_components':{'indices':cc_mask,'priority':masks_priorities[0]}, 
                    'reconstruction_error':{'indices':rec_mask,'priority':masks_priorities[1]}, 
                    'percentile_low':{'indices':prc_low_mask,'priority':masks_priorities[2]},
                    'percentile_high':{'indices':prc_high_mask,'priority':masks_priorities[3]},
                   }
    
    m_mask = get_merged_mask(weights_matrix = weights_matrix, 
                             masks_config = masks_config, 
                             max_base_pct = max_base_percentile, 
                             max_tail_pct = max_tail_percentile, 
                             tail_threshold = tail_threshold, 
                             verbose = verbose)
    return m_mask

## Генерация данных для НЧ

In [5]:
class NanoparticleDataFactory:
    def __init__(self, energy_grid:np.ndarray=None):
        '''
        energy_grid: общая сетка энергий (ось X) для интерполяции.
        Если None, создается стандартная сетка (np.arange(-10, 100.1, 0.5)).
        '''
        self.energy_grid = energy_grid if energy_grid is not None else np.arange(-10, 100.1, 0.5)
        self.basis_spectra = None  # Matrix (M_subs x E_points)
        self.basis_metadata = None # DataFrame: [sub_id, cn_vector, n_atoms]
        self.substructure_ids = []

    @staticmethod
    def get_coords_number_from_atom(atom:Atoms, n_spheres:int=7)->np.ndarray:
        '''
        Расчет CN для n_spheres сфер относительно поглощающего атома (индекс 0).
        Используется логика дискретных радиусов идеального кристалла.
        Метод возвращает векторв длины n_spheres со значениями соответствующих координационных чисел.
        '''
        pos = atom.get_positions()
        relative_coords = pos - pos[0]
        radii_2 = np.round((relative_coords**2).sum(axis=1), 3)
        
        unique_radii, counts = np.unique(radii_2, return_counts=True)
        
        neighbor_counts = counts[1:]
        
        template = np.zeros(n_spheres)
        valid_len = min(len(neighbor_counts), n_spheres)
        template[:valid_len] = neighbor_counts[:valid_len]
        return template

    @staticmethod
    def get_community_split(weights_matrix: np.ndarray, 
                            resolution: float = 2.0, 
                            threshold: float = 0.75, 
                            guaranteed_train_samples_idx: np.ndarray = None, 
                            sub_sizes: np.ndarray = None, 
                            seed: int = 42,
                            return_shared_idx:bool=False,
                           ) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
        '''
        Разделяет объекты (НЧ) и подструктуры на валидационные пулы (Train/Test) на основе 
        кластеризации графа смежности подструктур методом Louvain.
        
        Алгоритм изолирует подструктуры, минимизируя их пересечение между Train и Test.
        Объекты, чья зависимость от любого изолированного сообщества подструктур превышает "threshold",
        гарантированно отправляются в Test-выборку.
    
        Args:
            weights_matrix - Двумерный массив весов (объекты х подструктуры)
                             (не нормированная, сумма по axis=1 дает число атомов в НЧ).
            resolution - Параметр разрешения для алгоритма Louvain. Более высокие значения 
                приводят к обнаружению большего числа мелких сообществ. По умолчанию 2.0.
            threshold - Порог максимальной доли вовлеченности объекта в одно сообщество (0.0 - 1.0).
                Если объект привязан к какому-то сообществу сильнее этого порога, он уходит в Test. 
                По умолчанию 0.75.
            guaranteed_train_samples_idx (optional) - Индексы объектов, которые 
                жестко закреплены за Train-выборкой. По умолчанию None.
            sub_sizes (optional) - Одномерный массив размеров подструктур для 
                валидации распределений. По умолчанию None.
            seed - Сид для воспроизводимости случайных разбиений Louvain. По умолчанию 42.
            return_shared_idx - флаг для возврата индексов пересекающихся подструктур
    
        Returns:
            Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]: Кортеж, содержащий:
                - train_sample_idx (np.ndarray): Индексы объектов/НЧ для Train-выборки.
                - test_sample_idx (np.ndarray): Индексы объектов/НЧ для Test-выборки.
                - train_sub_indices (np.ndarray): Индексы подструктур, засвеченных в Train.
                - test_sub_indices (np.ndarray): Индексы подструктур, изолированных в Test
                                                 (не пересекающихся с Train).
                - shared_sub_indices (np.ndarray) если return_shared_idx=True
        '''
        
        if sub_sizes is not None:
            assert weights_matrix.shape[1] == sub_sizes.shape[0], \
                'sub_sizes.shape[0] должен совпадать с weights_matrix.shape[1]'    
                
        np_sizes = weights_matrix.sum(axis=1).astype(int)
        n_samples, n_subs = weights_matrix.shape
        binary_matrix = (weights_matrix > 0).astype(float)
    
        # Выделяем "остаток" данных для Louvain-кластеризации
        if guaranteed_train_samples_idx is not None:
            remaining_samples = np.setdiff1d(np.arange(n_samples), guaranteed_train_samples_idx)
        else:
            remaining_samples = np.arange(n_samples)
        
        if len(remaining_samples) == 0:
            return (guaranteed_train_samples_idx, np.array([], dtype=int), 
                    np.arange(n_subs), np.array([], dtype=int))
        
        # Строим граф смежности ТОЛЬКО для оставшихся объектов
        remaining_binary = binary_matrix[remaining_samples, :]
        adj = remaining_binary.T @ remaining_binary
        np.fill_diagonal(adj, 0)
        
        G = nx.from_numpy_array(adj)
        communities = nx.community.louvain_communities(G, seed=seed, resolution=resolution)
        node_groups = [list(c) for c in communities if len(c) > 0]
        
        if len(node_groups) == 0:
            node_groups = [[i] for i in range(n_subs)]
        
        # Расчет вкладов Louvain-сообществ в оставшиеся образцы
        division_g = np.array([remaining_binary[:, g].sum(axis=1) for g in node_groups])
        col_sums = division_g.sum(axis=0)
        col_sums[col_sums == 0] = 1
        division_g /= col_sums
    
        # Разделение оставшихся данных на Train/Test по порогу доминирования сообщества
        rem_train_idx = np.where(division_g.max(axis=0) < threshold)[0]
        louvain_train_samples = remaining_samples[rem_train_idx]
        louvain_test_samples = np.setdiff1d(remaining_samples, louvain_train_samples)
        
        # Итоговые пулы объектов
        if guaranteed_train_samples_idx is not None:
            train_sample_idx = np.unique(np.concatenate([guaranteed_train_samples_idx, louvain_train_samples]))
        else:
            train_sample_idx = louvain_train_samples
        test_sample_idx = louvain_test_samples
        
        # Разделение подструктур
        train_sub_indices = np.where(weights_matrix[train_sample_idx].sum(axis=0) > 0)[0]
        # Собираем ВСЕ подструктуры, которые физически присутствуют в Test-объектах
        if len(test_sample_idx) > 0:
            raw_test_subs = np.where(weights_matrix[test_sample_idx].sum(axis=0) > 0)[0]
        else:
            raw_test_subs = np.array([], dtype=int)
            
        shared_sub_indices = np.intersect1d(train_sub_indices, raw_test_subs)
        test_sub_indices = np.setdiff1d(raw_test_subs, train_sub_indices)

        
        train_sub_set = set(train_sub_indices)
        test_sub_set = set(test_sub_indices)
        shared_subs = set(shared_sub_indices)

        # Расчет статистик
        print('=' * 80)
        print('СТАТИСТИКА СООБЩЕСТВ И РАЗБИЕНИЯ ВЫБОРКИ:')
        print(f"Объектов в Train: {len(train_sample_idx)} | Объектов в Test: {len(test_sample_idx)}")
        print(f"Размер базиса подструктур: {n_subs} | Пересекающиеся подструктуры: {len(shared_subs)} ({len(shared_subs)/n_subs:.1%})")
        print(f"Уникально подструктур в Train: {len(train_sub_set - shared_subs)} | Уникально в Test: {len(test_sub_set - shared_subs)}")
        
        if len(test_sample_idx) > 0:
            train_np_sizes = np_sizes[train_sample_idx]
            test_np_sizes = np_sizes[test_sample_idx]
            
            edge_quantile = 0.03
            q_low, q_high = np.quantile(train_np_sizes, [edge_quantile, 1-edge_quantile])
            
            critical_small_in_test = int(np.sum(test_np_sizes < q_low))
            critical_large_in_test = int(np.sum(test_np_sizes > q_high))   
            edges_passed = (np.min(train_np_sizes) <= np.min(test_np_sizes)) and (np.max(train_np_sizes) >= np.max(test_np_sizes))
            fraction_inside = np.mean((test_np_sizes >= q_low) & (test_np_sizes <= q_high))
            
            print(f"Доля Test выборки внутри интервала [{edge_quantile},{1-edge_quantile}] Train данных: {fraction_inside:.3f}")
            print(f"Кол-во краевых малых НЧ в Test выборке (ниже {edge_quantile} квантиля Train): {critical_small_in_test}")
            print(f"Кол-во краевых крупных НЧ в Test выборке (выше {1-edge_quantile} квантиля Train): {critical_large_in_test}")
            print(f"Находятся ли краевые значения Test данных полностью внутри Train данных: {edges_passed}")
            
            np_ks_stat_ts, np_p_value_ts = ks_2samp(train_np_sizes, test_np_sizes, alternative='two-sided', method='asymp')
            np_ks_stat_gr, np_p_value_gr = ks_2samp(train_np_sizes, test_np_sizes, alternative='greater', method='asymp')
            print(f"Тест Колмогорова-Смирнова  для размеров НЧ (Train==Test): {np_ks_stat_ts:.5f} | P-value: {np_p_value_ts:.5f}")
            print(f"Тест Колмогорова-Смирнова  для размеров НЧ (Train>=Test): {np_ks_stat_gr:.5f} | P-value: {np_p_value_gr:.5f}")
            
            if sub_sizes is not None and len(train_sub_indices) > 0 and len(test_sub_indices) > 0:
                train_sub_sizes = sub_sizes[train_sub_indices]
                test_sub_sizes = sub_sizes[test_sub_indices]
                sub_ks_stat_ts, sub_p_value_ts = ks_2samp(train_sub_sizes, test_sub_sizes, alternative='two-sided', method='asymp')
                sub_ks_stat_gr, sub_p_value_gr = ks_2samp(train_sub_sizes, test_sub_sizes, alternative='greater', method='asymp')
                print(f"Тест Колмогорова-Смирнова  для размеров НК (Train==Test): {sub_ks_stat_ts:.5f} | P-value: {sub_p_value_ts:.5f}")
                print(f"Тест Колмогорова-Смирнова  для размеров НК (Train>=Test): {sub_ks_stat_gr:.5f} | P-value: {sub_p_value_gr:.5f}")
        else:
            print('ВНИМАНИЕ: Тестовая выборка пуста! Измените параметры resolution или threshold.')
        print('=' * 80)

        if return_shared_idx:
            return train_sample_idx, test_sample_idx, train_sub_indices, test_sub_indices, shared_sub_indices
        return train_sample_idx, test_sample_idx, train_sub_indices, test_sub_indices

    def load_basis(self, spectra_dir:Union[Path, str], substruct_df:pd.DataFrame, n_spheres:int=7):
        '''
        Загрузка спектров из файлов {index.txt} и расчет CN векторов.
        substruct_df: ['index', 'Structure', 'Num_atoms']
        Метод заполняет/создает поля: self.basis_spectra, self.basis_metadata, self.substructure_ids.
        '''
        path = Path(spectra_dir)
        spectra_list, metadata_list = [], []
        indices = substruct_df.index.tolist()

        for idx in indices:
            row = substruct_df.loc[idx]
            file_path = Path(path, f"{idx}.txt")
            
            if file_path.exists():
                data = np.loadtxt(file_path)
                f_interp = interp1d(data[:, 0], data[:, 1], bounds_error=False, fill_value=0)
                spectra_list.append(f_interp(self.energy_grid))
                
                cn_vec = self.get_coords_number_from_atom(row['Structure'], n_spheres=n_spheres)
                metadata_list.append({'sub_id': idx, 
                                      'n_atoms': row['Num_atoms'], 
                                      'cn_vector': cn_vec
                                     })
        
        self.basis_spectra = np.array(spectra_list)
        self.basis_metadata = pd.DataFrame(metadata_list)
        self.substructure_ids = self.basis_metadata['sub_id'].tolist()
        print(f"Готово. В базисе {len(self.substructure_ids)} подструктур.")

    def _apply_augmentation(self, raw_spectra:np.ndarray, params:dict)->np.ndarray:
        '''
        Физическая аугментация: сдвиг и растяжение оси E, гауссов шум.
        Возвращает новый спектр (np.ndarray)
        params: {scale (stratch); shift (eV); noise (% - 1==100%)}
        '''
        if not params: return raw_spectra
        raw_spectra = raw_spectra if len(raw_spectra.shape)==2 else raw_spectra.reshape(1,-1)
        
        aug_spectra = np.zeros_like(raw_spectra)
        e_grid = self.energy_grid
        
        for i, spec in enumerate(raw_spectra):
            s_val = np.random.uniform(1 - params.get('scale', 0), 1 + params.get('scale', 0))
            shift_val = np.random.uniform(-params.get('shift', 0), params.get('shift', 0))
            
            # Растяжение + сдвиг шкалы
            e_aug = e_grid * s_val + shift_val
            processed = np.interp(e_grid, e_aug, spec)
            
            # Шум
            if params.get('noise', 0) > 0:
                processed *= np.random.normal(1, params['noise'], len(processed))
            aug_spectra[i] = processed
        return aug_spectra

    def create_synthetic_mix(self, 
                             n_samples:int, 
                             k_range:Tuple[int,int]=(3, 10), 
                             pool_indices:np.ndarray=None)->np.ndarray:
        '''
        Генерация весов синтетических смесей из пула разрешенных подструктур.
        (Веса берутся из распределения Дирихле)
        n_samples - число образцов для генерации;
        k_range - диапазон допустимого числа подструктур для смешивания (3,10);
        pool_indices - список допустимых индексов, доступных для смешивания (None).
        '''
        m = self.basis_spectra.shape[0]
        indices = pool_indices if pool_indices is not None else np.arange(m)
        
        new_weights = np.zeros((n_samples, m))
        for i in range(n_samples):
            k = np.random.randint(k_range[0], k_range[1] + 1)
            k = k if k<len(indices) else len(indices)
            # Выбираем k случайных подструктур из пула
            chosen = np.random.choice(indices, k, replace=False)
            w = np.random.dirichlet(np.ones(len(chosen)))
            new_weights[i, chosen] = w
        return new_weights

    def create_perturbed_weights(self, 
                                 base_weights:np.ndarray, 
                                 sigma:float=0.05, 
                                 dropout_p:float=0.0)->np.ndarray:
        '''
        Возмущение матрицы весов base_weights случайным нормально распределенным шумом с std=sigma.
        Функция возвращает новую матрицу весов.
        dropout_p: опциональная вероятность обнуления очень малых вкладов (< 0.01).
        '''
        noise = np.random.normal(0, sigma, base_weights.shape)
        perturbed = base_weights + noise * (base_weights > 0)
        perturbed = np.maximum(perturbed, 0)
        
        if dropout_p > 0:
            # Обнуляем только очень маленькие веса и только с заданной вероятностью
            mask = (perturbed < 0.01) & (np.random.rand(*perturbed.shape) < dropout_p)
            perturbed[mask] = 0

        row_sums = perturbed.sum(axis=1, keepdims=True)
        row_sums[row_sums == 0] = 1
        return perturbed / row_sums

        
    def create_substitution(self, 
                            base_weights:np.ndarray, 
                            p:float=0.2, 
                            n_delta:int=5,
                            soft_split:bool=True)->np.ndarray:
        '''
        Замена подструктур в весах на близкие по числу атомов.
        Функция возвращает новую матрицу весов.
        p - вероятность замены (по умолчанию 0.2);
        n_delta - допустимый диапазон для отбора схожих подструктур по числу атомов в составе;
        soft_split=True: часть веса остается у оригинала, часть уходит кандидату;
        soft_split=False: жесткая замена другой близкой подструктурой.
        '''
        new_weights = base_weights.copy()
        meta = self.basis_metadata
        
        for i in range(len(new_weights)):
            if np.random.rand() < p:
                active = np.where(new_weights[i] > 0)[0]
                if len(active) == 0: continue
                
                idx_old = np.random.choice(active)
                n_atoms_old = meta.loc[idx_old]['n_atoms']
                
                # Поиск кандидатов в диапазоне n_atoms
                candidates = meta[(meta['n_atoms'] >= n_atoms_old - n_delta) & 
                                  (meta['n_atoms'] <= n_atoms_old + n_delta) & 
                                  (meta.index != idx_old)].index
                
                if len(candidates) > 0:
                    idx_new = np.random.choice(candidates)
                    current_weight = new_weights[i, idx_old]
                    
                    if soft_split:
                        # Генерируем долю, которая перейдет новой подструктуре (от 0 до 1)
                        fraction = np.random.rand()
                        new_weights[i, idx_new] += current_weight * fraction
                        new_weights[i, idx_old] = current_weight * (1 - fraction)
                    else:
                        new_weights[i, idx_new] += current_weight
                        new_weights[i, idx_old] = 0
                        
        return new_weights

    def generate_experiment_data(self, weights_matrix:np.ndarray, aug_params:dict=None):
        '''
        Функция возвращает массивы спектров и координационных чисел для заданной матрицы весов weights_matrix.
        aug_params: {scale (stratch); shift (eV); noise (% - 1==100%)}, по умолчанию None.
        Вывод (X,y):
            X - спектры (weights_matrix[0] x E), 
            y - координационные числа для первых n_spheres (по умолчанию (weights_matrix[0] x 7)).
        '''
        X = self._apply_augmentation(weights_matrix @ self.basis_spectra, aug_params)
        
        basis_cn_matrix = np.stack(self.basis_metadata['cn_vector'].values)
        y = weights_matrix @ basis_cn_matrix
        
        return X, y


    def plot_augmentation_check(self, sample_weights:np.ndarray, aug_params:dict, n_variants:int=3):
        '''
        Визуальный контроль влияния аугментации на конкретный образец
        sample_weights - вектор представление НЧ в базисе уникальных подструктур
        aug_params - параметры аугментации спектра aug_params: {scale (stratch); shift (eV); noise (% - 1==100%)}
        n_variants - число случайных аугментаций спектра.
        '''
        # raw_spec = (sample_weights.reshape(1, -1) @ self.basis_spectra)[0]
        raw_spec = (sample_weights @ self.basis_spectra)
        plt.figure(figsize=(10, 5))
        plt.plot(self.energy_grid, raw_spec, 'k-', lw=2, label='Original')
        
        for i in range(n_variants):
            aug_spec = self._apply_augmentation(raw_spec.reshape(1, -1), aug_params)[0]
            plt.plot(self.energy_grid, aug_spec, alpha=0.5, label=f'Augmented variant {i+1}')
            
        plt.title(f"Augmentation Check (Shift: {aug_params.get('shift')}eV, Scale: {aug_params.get('scale')})")
        plt.xlabel("Energy (eV)")
        plt.ylabel("Intensity")
        plt.legend()
        plt.grid(True, alpha=0.2)
        plt.show()

    def save_dataset(self, X, y, path='train'):
        '''
        Сохранение сгенерированных данных в npy файлы.
        '''
        np.save(f'{path}_X.npy', X)
        np.save(f'{path}_y.npy', y)
        print(f"Данные сохранены: {prefix}_X.npy, {prefix}_y.npy")


In [6]:
class Filter(ABC):
    '''
    Интерфейс для фильтрации сгенерированных данных
    '''
    @abstractclassmethod
    def check(self, cn_vector:np.ndarray, weight_vector:np.ndarray)->bool:
        '''
        cn_vector - вектор координационных чисел (n-e число соответствует n-ой координационной сфере)
        weight_vector - вектор представление НЧ в базисе подструктур.
        Возвращает True если фильтр пройден.
        '''
        pass

class DummyFilter(Filter):
    '''
    Фильтр всегда возвращает True
    '''
    def check(self, cn_vector:np.ndarray, weight_vector:np.ndarray)->bool:
        return True

class MinMaxFilter(Filter):
    '''
    Фильтр проверяет что данные cn_vectors находятся в интервалах от 0 до max_cn_values включительно.
    '''
    def __init__(self, max_cn_values:np.ndarray=np.array([12, 6, 24, 12, 24, 8, 48])):
        self.min_cn_values = np.zeros_like(max_cn_values)
        self.max_cn_values = max_cn_values
    
    def check(self, cn_vector:np.ndarray, weight_vector:np.ndarray)->bool:
        len_cn = len(cn_vector)
        if len_cn<=len(self.max_cn_values):
            return np.all(cn_vector>=self.min_cn_values[:len_cn])&np.all(cn_vector<=self.max_cn_values[:len_cn])
        else:
            return False
            
class FCC_BulkCutFilter(Filter):
    '''
    Фильтр для проверки возможности "вырезания" такой частицы из фольги.
    При вырезании из идеального кристалл без добавления внутренних полостей
    внутренняя координационная сфера не может быть заполнена меньше чем внешняя.
    Данный фильтр смотрит на 1-ю и 2-ю координационные сферы.
    '''
    def check(self, cn_vector:np.ndarray, weight_vector:np.ndarray)->bool:
        if len(cn_vector)>1:
            return cn_vector[0]>=(cn_vector[1]*2)
        else:
            return False

class CompositeFilter(Filter):
    '''
    Объединяет список фильтров в один.
    '''
    def __init__(self, filters: List[Filter]):
        self.filters = filters

    def check(self, cn_vector: np.ndarray, weight_vector: np.ndarray) -> bool:
        return all(f.check(cn_vector, weight_vector) for f in self.filters)

In [7]:
class AdvancedDataGenerator:
    def __init__(self, 
                 factory:NanoparticleDataFactory, 
                 sigma:float=0.07, 
                 n_delta:int=4, 
                 aug_params:dict=None, 
                 data_filter:Filter=None):
        '''
        factory - Экземпляр NanoparticleDataFactory
        sigma - % возмущений для базиса
        n_delta - разница в атомах для подмены
        aug_params - параметры аугментации (сдвиг, шум и т.д.)
        data_filter - проинициализированный фильтр данных, 
        по умолчанию None (DummyFilter возвращающий всегда True) 
        '''
        self.factory = factory 
        self.sigma = sigma
        self.n_delta = n_delta
        self.aug_params = aug_params
        self.data_filter = data_filter if data_filter is not None else DummyFilter()

    def generate_synthetic_set(self, 
                               basis_indices:np.ndarray, 
                               n_samples:int=5000, 
                               k_mix:int=3, 
                               batch_size:int=1000,
                               return_weights:bool = False):
        '''
        Генерация синтетики с использованием логики Factory для создания смесей. 
        (X-спектры,y-вектора CN)
        basis_indices - массив допустимых для смешиванию индексов.
        n_samples - сколько нужно сгенерировать данных
        k_mix - сколько подструктур будут замешиваться для 1-й синтетической НЧ
        batch_size - размер батча для генерации данных
        return_weights - флаг для возврата новой матрицы весов. (X,y,weights)
        '''
        X, y, w_out = [], [], []
        
        pool_positions = basis_indices
        
        while len(X) < n_samples:
            remaining = n_samples - len(X)
            current_batch = min(batch_size, int(remaining * 1.5) + 10)
            
            weights_batch = self.factory.create_synthetic_mix(n_samples=current_batch, 
                                                              k_range=(k_mix, k_mix), 
                                                              pool_indices=pool_positions
                                                             )

            for w_vec in weights_batch:
                if len(X) >= n_samples:
                    break
                
                spectrum, cn_vec = self.factory.generate_experiment_data(weights_matrix=w_vec, 
                                                                         aug_params=self.aug_params
                                                                        )
                
                # Страховка размерности
                spectrum = np.asarray(spectrum).flatten()
                
                # Фильтрация данных
                if self.data_filter.check(cn_vector=cn_vec, weight_vector=w_vec):
                    X.append(spectrum)
                    y.append(cn_vec)
                    if return_weights:
                        w_out.append(w_vec)

        if return_weights:
            return np.array(X), np.array(y), np.array(w_out)       
        return np.array(X), np.array(y)

    def generate_honest_set(self, 
                            base_weights:np.ndarray, 
                            allowed_indices:np.ndarray=None, 
                            n_samples_target:int=None, 
                            p_subst:float=0.2, 
                            return_weights:bool = False):
            '''
            Генерация на основе реальных НЧ с опциональным возвратом весов подструктур. 
            (X-спектры,y-вектора CN)
            base_weights - матрица весов подструктур для исходных НЧ.
            allowed_indices - индексы допустимых для подмены подструктур.
            n_samples_target - сколько объектов надо сгенерировать.
            p_subst - вероятность подмены подструктуры.
            return_weights - флаг для возврата новой матрицы весов. (X,y,weights)
            '''
            X, y, w_out = [], [], []
            # Сохраняем оригинал метаданных
            original_meta = self.factory.basis_metadata.copy()
            
            # Целевое количество
            target = n_samples_target if n_samples_target else len(base_weights) * 5
            
            try:
                if allowed_indices is not None:
                    # Временная модификация метаданных для ограничения пула кандидатов на замену
                    mask = self.factory.basis_metadata.index.isin(allowed_indices)
                    temp_meta = self.factory.basis_metadata.copy()
                    temp_meta.loc[~mask, 'n_atoms'] = -999999  # Исключаем из поиска по n_delta
                    self.factory.basis_metadata = temp_meta
    
                while len(X) < target:
                    # Зашумление в весах оригинальных НЧ
                    perturbed = self.factory.create_perturbed_weights(base_weights=base_weights, 
                                                                      sigma=self.sigma, 
                                                                      dropout_p=0.0
                                                                     )
                    
                    # Замена подструктур похожими по размеру (n_delta)
                    final_weights = self.factory.create_substitution(base_weights=perturbed, 
                                                                     p=p_subst, 
                                                                     n_delta=self.n_delta, 
                                                                     soft_split=True
                                                                    )
                    
                    # Генерация спектров
                    for w_vec in final_weights:
                        if len(X) >= target: break
                        
                        # Генерация спектра и расчет взвешенного CN
                        spectrum, cn_vec = self.factory.generate_experiment_data(
                            weights_matrix=w_vec, 
                            aug_params=self.aug_params
                        )
                        
                        spectrum = np.asarray(spectrum).flatten()
                        
                        # Фильтрация данных
                        if self.data_filter.check(cn_vector=cn_vec, weight_vector=w_vec):
                            X.append(spectrum)
                            y.append(cn_vec)
                            if return_weights:
                                w_out.append(w_vec)
                                
            finally:
                # Гарантированный возврат метаданных в исходное состояние
                self.factory.basis_metadata = original_meta
                
            if return_weights:
                return np.array(X), np.array(y), np.array(w_out)
            return np.array(X), np.array(y)

In [8]:
def select_strict_basis_consistent(df_subs:pd.DataFrame, n_target:int=200, train_ratio:float=0.8)->Tuple[np.ndarray,np.ndarray]:
    '''
    Выбирает базис уникальных подструктур так, чтобы и Train, и Test содержали структуры
    во всем диапазоне размеров (Num_atoms), обеспечивая отсутствие значимого сдвига данных.
    df_subs: ['index', 'Structure', 'Num_atoms']
    '''
    train_indices = []
    test_indices = []
    
    # Обязательно фиксируем края для трейна
    idx_min = df_subs['Num_atoms'].idxmin()
    idx_max = df_subs['Num_atoms'].idxmax()
    train_indices.extend([idx_min, idx_max])
    
    # Группируем остальные структуры по количеству атомов
    remaining_df = df_subs.drop(index=[idx_min, idx_max])
    grouped = remaining_df.groupby('Num_atoms')
    
    for size, group in grouped:
        indices = group.index.tolist()
        if len(indices) >= 2:
            # Если есть хотя бы 2 структуры этого размера, делим их
            # (одна гарантированно в трейн, одна гарантированно в тест)
            np.random.shuffle(indices)
            test_indices.append(indices.pop())
            train_indices.append(indices.pop())
        else:
            # Если структура уникального размера всего одна, отдаем в трейн
            train_indices.append(indices.pop())
            
    # Добираем до n_target, соблюдая пропорцию и не пересекая сеты.
    current_total = len(train_indices) + len(test_indices)
    needed = n_target - current_total
    
    if needed > 0:
        # Пул еще не использованных структур
        used_so_far = set(train_indices) | set(test_indices)
        pool_indices = df_subs.index.difference(used_so_far).tolist()
        np.random.shuffle(pool_indices)
        
        # Определяем, сколько из "добавочных" пойдет в тест, чтобы выровнять общую пропорцию
        target_test_total = int(n_target * (1 - train_ratio))
        test_needed = max(0, target_test_total - len(test_indices))
        
        # Заполняем тест, остальное в трейн
        test_indices.extend(pool_indices[:test_needed])
        train_indices.extend(pool_indices[test_needed : needed])
        
    # Ограничиваем, если вдруг перебрали
    return np.array(train_indices[:int(n_target*train_ratio)]), np.array(test_indices[:int(n_target*(1-train_ratio))])


## Визуализация

In [9]:
def plot_dataset_variability(X_data, energy_grid, foil_spectrum=None, title_suffix='Generated Dataset', save_name:str=None):
    '''
    Визуализация статистического разброса спектров в датасете.
    
    X_data: np.ndarray (N_samples x E_points) — сгенерированные спектры.
    energy_grid: np.ndarray — ось энергии (X).
    foil_spectrum: np.ndarray (optional) — спектр фольги (Y) для сравнения.
    '''
    mean_y = np.mean(X_data, axis=0)
    std_y = np.std(X_data, axis=0)
    min_y = np.min(X_data, axis=0)
    max_y = np.max(X_data, axis=0)
    
    lower_std = mean_y - std_y
    upper_std = mean_y + std_y

    plt.figure(figsize=(10, 6))
    
    plt.fill_between(energy_grid, min_y, max_y, color='gray', alpha=0.15, label='Total Range (min-max)')
    
    plt.fill_between(energy_grid, lower_std, upper_std, color='royalblue', alpha=0.35, label='Std Variance (±1σ)')
    
    plt.plot(energy_grid, mean_y, color='mediumblue', linewidth=2.5, label='Mean Synthetic Spectrum')

    if foil_spectrum is not None:
        C = np.sum(foil_spectrum * mean_y) / np.sum(foil_spectrum**2)
        plt.plot(energy_grid, foil_spectrum * C, color='black', ls='--', lw=1.5, 
                 label='Foil Reference (Normalized)', alpha=0.8)

    title = f"XANES Variability Analysis: {title_suffix} (N={X_data.shape[0]})"
    plt.title(title, fontsize=12)
    plt.xlabel('Energy (eV)', fontsize=11)
    plt.ylabel('Normalized Absorption (arb. units)', fontsize=11)

    plt.xlim(-10, 60)
    
    plt.legend(loc='upper right', frameon=True, shadow=True)
    plt.grid(True, linestyle=':', alpha=0.6)

    if save_name:
        plt.savefig(f"{save_name}.png", dpi=600, bbox_inches='tight')
    
    plt.show()

def plot_cn_distribution(y_matrix, save_name:str=None):
    '''
    Гистограммы CN для каждой сферы.
    '''
    n_spheres = y_matrix.shape[1]
    fig, axes = plt.subplots(1, n_spheres, figsize=(3 * n_spheres, 3))
    for i in range(n_spheres):
        axes[i].hist(y_matrix[:, i], bins=25, color='steelblue', alpha=0.7)
        axes[i].set_title(f"Sphere {i+1}")
    plt.tight_layout()
    
    if save_name:
        plt.savefig(f"{save_name}.png", dpi=600, bbox_inches='tight')
    
    plt.show()

In [10]:
def plot_substructure_audit(weights_matrix, train_sample_idx, test_sample_idx, metric='dice', save_name:str=None):
    # Находим маску подструктур, которые присутствуют ХОТЯ БЫ у одного объекта
    active_subs_mask = np.any(weights_matrix > 0, axis=0)
    # Подготовка данных
    is_in_train = np.any(weights_matrix[train_sample_idx] > 0, axis=0)[active_subs_mask]
    is_in_test = np.any(weights_matrix[test_sample_idx] > 0, axis=0)[active_subs_mask]
    
    # categories должен иметь длину, равную количеству АКТИВНЫХ подструктур
    n_active_subs = np.sum(active_subs_mask)
    categories = np.zeros(n_active_subs, dtype=int)
    
    # categories = np.zeros(weights_matrix.shape[1], dtype=int)
    categories[(is_in_train) & (~is_in_test)] = 0 # Train Only
    categories[(is_in_test) & (~is_in_train)] = 1 # Test Only
    categories[(is_in_train) & (is_in_test)] = 2  # Common

    # Подготовка матрицы для UMAP
    # Превращаем в бинарный вид и добавляем микро-шум для стабилизации
    data_to_embed = (weights_matrix[:, active_subs_mask].T > 0).astype(float)
    # data_to_embed = (weights_matrix.T > 0).astype(float)
    # Добавляем исчезающе малый шум, чтобы убрать идеальную разреженность
    # rng = np.random.default_rng(42)
    # data_to_embed += rng.normal(0, 1e-6, data_to_embed.shape)

    print(f"Всего подструктур в базисе: {weights_matrix.shape[1]}")
    print(f"Активных подструктур для UMAP: {n_active_subs} (пустые {weights_matrix.shape[1] - n_active_subs} отброшены)")

    # metric='dice', 'cosine', 'jaccard'
    reducer = umap.UMAP(n_neighbors=30, 
                        min_dist=0.1, 
                        metric=metric,  #dice
                        init='random', 
                        # random_state=42, 
                        n_epochs=500
                       )
    
    print('Запуск UMAP (может занять время)...')
    embedding = reducer.fit_transform(data_to_embed)

    plt.figure(figsize=(10, 8))
    
    colors = ['#3498db', '#e74c3c', '#f1c40f']
    names = ['Train Only', 'Test Only', 'Common']
    
    for i in [0, 2, 1]:  # Порядок слоев
        mask = categories == i
        if not np.any(mask): continue # Проверка, если какая-то категория пуста
            
        plt.scatter(
            embedding[mask, 0], 
            embedding[mask, 1], 
            c=colors[i], 
            label=f"{names[i]} (n={mask.sum()})",
            s=25 if i != 1 else 50, # Тест покрупнее
            alpha=0.6,
            edgecolors='white' if i == 1 else 'none',
            linewidth=0.5
        )

    plt.title('Audit of Data Splitting & Substructure Overlap')
    plt.legend()
    plt.grid(True, alpha=0.2)
    if save_name:
        plt.savefig(f"{save_name}.png", dpi=600, bbox_inches='tight')
    plt.show()